In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

def generate_balanced_adversarial_proxy(csv_path: str, 
                                        output_path: str = None, 
                                        target_f1: float = 0.34, 
                                        new_col_name: str = "ML1_proxy_adv_f1_34",
                                        random_seed: int = 42):
    np.random.seed(random_seed)
    
    print(f"📂 正在读取文件: {csv_path}...")
    df = pd.read_csv(csv_path)
    
    oracle_col = "ML1_oracle2_probability"
    if oracle_col not in df.columns:
        raise ValueError(f"未在 CSV 中找到列: {oracle_col}")
    
    y_true = (df[oracle_col] > 0.5).astype(int).values
    pos_indices = np.where(y_true == 1)[0]
    neg_indices = np.where(y_true == 0)[0]
    
    P = len(pos_indices)
    N = len(neg_indices)
    print(f"📊 样本总数: {len(df)} | 真实正例数(P): {P} | 真实负例数(N): {N}")
    
    # 🎯 目标：使得 F1 贴近 target_f1 的同时，让 Precision 和 Recall 尽可能均衡（都较低）
    best_loss = float("inf")
    best_tp, best_fp = None, None
    
    for tp in range(1, P + 1):
        fp = int(round(((2.0 - target_f1) / target_f1) * tp - P))
        if 0 <= fp <= N:
            current_f1 = (2.0 * tp) / (P + tp + fp)
            current_prec = tp / (tp + fp)
            current_rec = tp / P
            
            # 损失函数：主要保证 F1 准确，次要保证 Precision 与 Recall 平衡（差值最小）
            f1_error = abs(current_f1 - target_f1)
            balance_error = abs(current_prec - current_rec)
            total_loss = f1_error * 100.0 + balance_error
            
            if total_loss < best_loss:
                best_loss = total_loss
                best_tp, best_fp = tp, fp

    best_fn = P - best_tp
    best_tn = N - best_fp
    
    print(f"\n🎯 均衡对抗设定 (Target F1 ≈ {target_f1}):")
    print(f"   - TP (真阳性 - 保留的真信号) : {best_tp}")
    print(f"   - FP (假阳性 - 对抗性误报)   : {best_fp}")
    print(f"   - FN (假阴性 - 对抗性漏报)   : {best_fn}")
    print(f"   - TN (真阴性 - 正常负样本)   : {best_tn}")

    # 随机打乱正负样本索引
    shuffled_pos = np.random.permutation(pos_indices)
    shuffled_neg = np.random.permutation(neg_indices)
    
    tp_idx = shuffled_pos[:best_tp]
    fn_idx = shuffled_pos[best_tp:]
    fp_idx = shuffled_neg[:best_fp]
    tn_idx = shuffled_neg[best_fp:]
    
    # 生成平滑的对抗性概率（Beta分布模拟真实置信度）
    proxy_probs = np.zeros(len(df), dtype=float)
    
    # TP (真阳性): 预测 > 0.5
    proxy_probs[tp_idx] = np.random.beta(a=4, b=2, size=len(tp_idx)) * 0.48 + 0.51  # [0.51, 0.99]
    # FP (假阳性): 预测 > 0.5 (大量注入欺骗信号)
    proxy_probs[fp_idx] = np.random.beta(a=3, b=2, size=len(fp_idx)) * 0.46 + 0.52  # [0.52, 0.98]
    # FN (假阴性): 预测 <= 0.5 (大量漏检)
    proxy_probs[fn_idx] = np.random.beta(a=2, b=4, size=len(fn_idx)) * 0.46 + 0.02  # [0.02, 0.48]
    # TN (真阴性): 预测 <= 0.5
    proxy_probs[tn_idx] = np.random.beta(a=1.5, b=5, size=len(tn_idx)) * 0.47 + 0.01 # [0.01, 0.48]

    df[new_col_name] = np.clip(proxy_probs, 0.0001, 0.9999).round(6)
    
    # 评测实际指标
    y_pred = (df[new_col_name] > 0.5).astype(int).values
    achieved_f1 = f1_score(y_true, y_pred)
    achieved_prec = precision_score(y_true, y_pred)
    achieved_rec = recall_score(y_true, y_pred)

    print("\n" + "=" * 50)
    print(f"✅ 生成完毕！列名: {new_col_name}")
    print(f"📊 实际指标评估 (阈值 > 0.5):")
    print(f"   - F1-Score : {achieved_f1:.4f} (期望: {target_f1})")
    print(f"   - Precision: {achieved_prec:.4f}")
    print(f"   - Recall   : {achieved_rec:.4f}")
    print("=" * 50)
    
    if output_path is None:
        output_path = csv_path
    df.to_csv(output_path, index=False)
    print(f"💾 已成功写回/保存至: {output_path}")

if __name__ == "__main__":
    generate_balanced_adversarial_proxy(
        csv_path="/home/wangshuo/resource/datasets/parler_data/dataset_three/csv_data/post.csv",
        output_path="/home/wangshuo/resource/datasets/parler_data/dataset_three/csv_data/post.csv", # 直接更新覆盖原文件
        target_f1=0.34,
        new_col_name="ML1_proxy_adv"
    )

📂 正在读取文件: /home/wangshuo/resource/datasets/parler_data/dataset_three/csv_data/post.csv...
📊 样本总数: 24082 | 真实正例数(P): 1825 | 真实负例数(N): 22257

🎯 均衡对抗设定 (Target F1 ≈ 0.34):
   - TP (真阳性 - 保留的真信号) : 621
   - FP (假阳性 - 对抗性误报)   : 1207
   - FN (假阴性 - 对抗性漏报)   : 1204
   - TN (真阴性 - 正常负样本)   : 21050

✅ 生成完毕！列名: ML1_proxy_adv
📊 实际指标评估 (阈值 > 0.5):
   - F1-Score : 0.3400 (期望: 0.34)
   - Precision: 0.3397
   - Recall   : 0.3403
💾 已成功写回/保存至: /home/wangshuo/resource/datasets/parler_data/dataset_three/csv_data/post.csv
